In [1]:
import pandas as pd

df = pd.read_excel('/kaggle/input/datasets/mgbleh/carbook/CarBook.xlsx')
df.head()

,Car_ID,Brand,Model,Year,Fuel_Type,Transmission,Price,Mileage,Engine_CC,Seating_Capacity,Service_Cost
0,1,Toyota,Innova,2024,CNG,Manual,2020000,27.3,800,4,24100
1,2,Kia,EV6,2023,Diesel,Manual,1770000,16.4,2500,7,18800
2,3,Maruti Suzuki,Dzire,2016,Petrol,Manual,3430000,17.6,2000,6,24700
3,4,Honda,Amaze,2019,Petrol,Manual,1610000,19.2,2500,6,23300
4,5,Honda,City,2015,Electric,Manual,1840000,15.8,1000,5,5800


In [2]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/mgbleh/carbook/CarBook.xlsx


In [3]:
import duckdb

duckdb.query("""
CREATE TABLE cars (
    Car_ID INTEGER,
    Brand TEXT,
    Model TEXT,
    Year INTEGER,
    Fuel_Type TEXT,
    Transmission TEXT,
    Price INTEGER,
    Mileage FLOAT,
    Engine_CC INTEGER,
    Seating_Capacity INTEGER,
    Service_Cost INTEGER
)
""")

In [4]:
import duckdb

duckdb.query("CREATE OR REPLACE TABLE cars AS SELECT * FROM df")

In [5]:
result = duckdb.query("SELECT * FROM cars LIMIT 5").to_df()
result

,Car_ID,Brand,Model,Year,Fuel_Type,Transmission,Price,Mileage,Engine_CC,Seating_Capacity,Service_Cost
0,1,Toyota,Innova,2024,CNG,Manual,2020000,27.3,800,4,24100
1,2,Kia,EV6,2023,Diesel,Manual,1770000,16.4,2500,7,18800
2,3,Maruti Suzuki,Dzire,2016,Petrol,Manual,3430000,17.6,2000,6,24700
3,4,Honda,Amaze,2019,Petrol,Manual,1610000,19.2,2500,6,23300
4,5,Honda,City,2015,Electric,Manual,1840000,15.8,1000,5,5800


In [6]:
import duckdb

query = """
SELECT Brand,
       ROUND(AVG(Price),0) AS avg_price
FROM cars
GROUP BY Brand
ORDER BY avg_price DESC
"""

result = duckdb.query(query).to_df()
result.head()

,Brand,avg_price
0,Mahindra,1996396.0
1,Honda,1972169.0
2,Volkswagen,1960909.0
3,Skoda,1955974.0
4,Renault,1950783.0


In [7]:
import duckdb

# 1️⃣ Average car price by brand
query1 = """
SELECT Brand,
       ROUND(AVG(Price),0) AS avg_price
FROM cars
GROUP BY Brand
ORDER BY avg_price DESC
"""
result1 = duckdb.query(query1).to_df()
print("1️⃣ Average car price by brand")
print(result1)


# 2️⃣ Average mileage by fuel type
query2 = """
SELECT Fuel_Type,
       ROUND(AVG(Mileage),2) AS avg_mileage
FROM cars
GROUP BY Fuel_Type
ORDER BY avg_mileage DESC
"""
result2 = duckdb.query(query2).to_df()
print("\n2️⃣ Average mileage by fuel type")
print(result2)


# 3️⃣ Most affordable car per brand (window function)
query3 = """
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER(PARTITION BY Brand ORDER BY Price) AS rn
    FROM cars
)
WHERE rn = 1
"""
result3 = duckdb.query(query3).to_df()
print("\n3️⃣ Most affordable car per brand")
print(result3)


# 4️⃣ Car count by transmission type
query4 = """
SELECT Transmission,
       COUNT(*) AS num_models
FROM cars
GROUP BY Transmission
"""
result4 = duckdb.query(query4).to_df()
print("\n4️⃣ Car count by transmission type")
print(result4)


# 5️⃣ Price vs Engine Category
query5 = """
SELECT
    CASE
        WHEN Engine_CC < 1200 THEN 'Small Engine'
        WHEN Engine_CC BETWEEN 1200 AND 1800 THEN 'Medium Engine'
        ELSE 'Large Engine'
    END AS engine_category,
    ROUND(AVG(Price),0) AS avg_price
FROM cars
GROUP BY engine_category
ORDER BY avg_price
"""
result5 = duckdb.query(query5).to_df()
print("\n5️⃣ Price vs Engine Category")
print(result5)


# 6️⃣ Top 10 most economical cars (high mileage, low price)
query6 = """
SELECT Brand, Model, Price, Mileage
FROM cars
ORDER BY Mileage DESC, Price ASC
LIMIT 10
"""
result6 = duckdb.query(query6).to_df()
print("\n6️⃣ Top 10 most economical cars")
print(result6)


# 7️⃣ Annual ownership cost over 5 years
query7 = """
SELECT Brand, Model,
       Price + (Service_Cost * 5) AS five_year_cost
FROM cars
ORDER BY five_year_cost ASC
LIMIT 10
"""
result7 = duckdb.query(query7).to_df()
print("\n7️⃣ Annual ownership cost over 5 years")
print(result7)


# 8️⃣ Average service cost by brand
query8 = """
SELECT Brand,
       ROUND(AVG(Service_Cost),0) AS avg_service_cost
FROM cars
GROUP BY Brand
ORDER BY avg_service_cost DESC
"""
result8 = duckdb.query(query8).to_df()
print("\n8️⃣ Average service cost by brand")
print(result8)


# 9️⃣ Cars with seating >= 7 and high mileage
query9 = """
SELECT Brand, Model, Seating_Capacity, Mileage
FROM cars
WHERE Seating_Capacity >= 7
ORDER BY Mileage DESC
"""
result9 = duckdb.query(query9).to_df()
print("\n9️⃣ Cars with seating >= 7 and high mileage")
print(result9)


# 🔟 Recent models with premium pricing (Year >= 2022, Price > 15 lakh INR)
query10 = """
SELECT Brand, Model, Year, Price
FROM cars
WHERE Year >= 2022 AND Price > 1500000
ORDER BY Price DESC
"""
result10 = duckdb.query(query10).to_df()
print("\n🔟 Recent models with premium pricing")
print(result10)

1️⃣ Average car price by brand
           Brand  avg_price
0       Mahindra  1996396.0
1          Honda  1972169.0
2     Volkswagen  1960909.0
3          Skoda  1955974.0
4        Renault  1950783.0
5         Toyota  1939549.0
6        Hyundai  1932594.0
7  Maruti Suzuki  1917831.0
8            Kia  1917307.0
9    Tata Motors  1916198.0

2️⃣ Average mileage by fuel type
  Fuel_Type  avg_mileage
0       CNG        20.10
1    Diesel        20.00
2    Petrol        19.91
3  Electric        19.85

3️⃣ Most affordable car per brand
   Car_ID          Brand     Model  Year Fuel_Type Transmission   Price  \
0     709          Honda      Jazz  2018    Petrol       Manual  400000   
1    4523    Tata Motors     Nexon  2024    Petrol    Automatic  400000   
2    1286          Skoda    Slavia  2020  Electric       Manual  400000   
3    8070  Maruti Suzuki     Swift  2023  Electric    Automatic  400000   
4    9189            Kia    Seltos  2021  Electric    Automatic  400000   
5    3389       M

In [8]:
import duckdb

# Top 10 most affordable cars based on Price + Service_Cost
query_affordable = """
SELECT Brand,
       Model,
       Price,
       Service_Cost,
       (Price + Service_Cost) AS total_cost
FROM cars
ORDER BY total_cost ASC
LIMIT 10
"""

result_affordable = duckdb.query(query_affordable).to_df()
print("Top 10 most affordable cars (Price + Service Cost)")
print(result_affordable)

Top 10 most affordable cars (Price + Service Cost)
         Brand          Model   Price  Service_Cost  total_cost
0       Toyota       Fortuner  400000          6500      406500
1       Toyota       Fortuner  400000         11200      411200
2       Toyota  Urban Cruiser  400000         11600      411600
3        Honda           Jazz  400000         12500      412500
4        Honda           Jazz  400000         13100      413100
5          Kia            EV6  400000         13300      413300
6  Tata Motors          Nexon  400000         14800      414800
7     Mahindra         XUV700  410000          5000      415000
8        Skoda         Slavia  410000          6100      416100
9          Kia       Carnival  400000         16300      416300


In [9]:
import duckdb

# 1️⃣ Top 3 most affordable cars per brand
query1 = """
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY Brand ORDER BY (Price + Service_Cost) ASC) AS brand_rank
    FROM cars
)
WHERE brand_rank <= 3
ORDER BY Brand, brand_rank
"""
result1 = duckdb.query(query1).to_df()
print("1️⃣ Top 3 most affordable cars per brand")
print(result1)


# 2️⃣ Cars with mileage above brand average
query2 = """
SELECT Brand, Model, Mileage
FROM cars
WHERE Mileage > (
    SELECT AVG(Mileage) 
    FROM cars AS c2
    WHERE c2.Brand = cars.Brand
)
ORDER BY Brand, Mileage DESC
"""
result2 = duckdb.query(query2).to_df()
print("\n2️⃣ Cars with mileage above brand average")
print(result2)


# 3️⃣ Price quartiles (segmentation)
query3 = """
SELECT *,
       NTILE(4) OVER (ORDER BY Price DESC) AS price_quartile
FROM cars
ORDER BY Price DESC
"""
result3 = duckdb.query(query3).to_df()
print("\n3️⃣ Price quartiles for cars")
print(result3)


# 4️⃣ Cumulative total cost by ascending affordability
query4 = """
SELECT Brand, Model, Price, Service_Cost,
       SUM(Price + Service_Cost) OVER (ORDER BY (Price + Service_Cost)) AS cumulative_cost
FROM cars
ORDER BY cumulative_cost
"""
result4 = duckdb.query(query4).to_df()
print("\n4️⃣ Cumulative total cost by affordability")
print(result4)


# 5️⃣ Average price by fuel type and transmission
query5 = """
SELECT Fuel_Type,
       Transmission,
       ROUND(AVG(Price),0) AS avg_price
FROM cars
GROUP BY Fuel_Type, Transmission
ORDER BY Fuel_Type, Transmission
"""
result5 = duckdb.query(query5).to_df()
print("\n5️⃣ Average price by fuel type and transmission")
print(result5)


# 6️⃣ Price categories: Budget / Mid-range / Premium
query6 = """
SELECT Brand, Model, Price,
       CASE
           WHEN Price < 500000 THEN 'Budget'
           WHEN Price BETWEEN 500000 AND 1500000 THEN 'Mid-range'
           ELSE 'Premium'
       END AS price_category
FROM cars
ORDER BY Price
"""
result6 = duckdb.query(query6).to_df()
print("\n6️⃣ Price categories for cars")
print(result6)


# 7️⃣ Top 5 most economical cars by fuel type
query7 = """
SELECT *
FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY Fuel_Type ORDER BY Mileage DESC, Price ASC) AS rn
    FROM cars
)
WHERE rn <= 5
ORDER BY Fuel_Type, rn
"""
result7 = duckdb.query(query7).to_df()
print("\n7️⃣ Top 5 most economical cars by fuel type")
print(result7)


# 8️⃣ Rank cars by total cost across all brands
query8 = """
SELECT Brand, Model, Price, Service_Cost,
       (Price + Service_Cost) AS total_cost,
       RANK() OVER (ORDER BY (Price + Service_Cost) ASC) AS affordability_rank
FROM cars
ORDER BY affordability_rank
LIMIT 10
"""
result8 = duckdb.query(query8).to_df()
print("\n8️⃣ Rank cars by total cost (top 10)")
print(result8)


# 9️⃣ Average service cost by brand, ordered descending
query9 = """
SELECT Brand,
       ROUND(AVG(Service_Cost),0) AS avg_service_cost
FROM cars
GROUP BY Brand
ORDER BY avg_service_cost DESC
"""
result9 = duckdb.query(query9).to_df()
print("\n9️⃣ Average service cost by brand")
print(result9)


# 🔟 Cars with engine > 1800 CC and high price
query10 = """
SELECT Brand, Model, Engine_CC, Price
FROM cars
WHERE Engine_CC > 1800 AND Price > 1500000
ORDER BY Price DESC
"""
result10 = duckdb.query(query10).to_df()
print("\n🔟 Cars with engine > 1800 CC and high price")
print(result10)

1️⃣ Top 3 most affordable cars per brand
    Car_ID          Brand          Model  Year Fuel_Type Transmission   Price  \
0      709          Honda           Jazz  2018    Petrol       Manual  400000   
1     4267          Honda           Jazz  2018    Diesel       Manual  400000   
2     9563          Honda          Amaze  2022       CNG    Automatic  420000   
3     6135        Hyundai          Creta  2020    Petrol    Automatic  400000   
4      993        Hyundai          Venue  2017  Electric    Automatic  410000   
5     6647        Hyundai            i20  2022    Petrol       Manual  420000   
6     1504            Kia            EV6  2020    Petrol    Automatic  400000   
7     3617            Kia       Carnival  2016    Petrol       Manual  400000   
8     9942            Kia         Carens  2020       CNG    Automatic  410000   
9     1202       Mahindra         XUV700  2017  Electric    Automatic  410000   
10    7277       Mahindra           Thar  2021  Electric    Automati